# Stage 5: Hyperparameter Tuning with weighted Purged CV

Randomized search over a `PurgedKFold` splitter, **weighted** at both fit and score time. Trial scores are summarised, the best trial's params are saved, and a Deflated-Sharpe-style multiple-testing deflator is applied to the trial pool.

**About DSR here**: AFML defines DSR for *strategy returns*. The value computed in this notebook is the same formula applied to the **trial CV-Sharpe statistics** `mean(fold_scores) / std(fold_scores)`, i.e. an in-sample multiple-testing correction across hyperparameter trials. It is *not* a statement about live PnL Sharpe — that belongs to the backtesting stage.

In [ ]:
import sys, os, json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.ensemble import RandomForestClassifier
from xgboost import XGBClassifier

sys.path.insert(0, os.path.abspath('../..'))
from src.cross_validation import PurgedKFold, cv_score
from src.hyperparameter_tuning import (
    purged_random_search,
    deflated_sharpe_ratio_for_trials,
)

plt.style.use('seaborn-v0_8-darkgrid')
RNG = 42

## 1. Load dataset

In [ ]:
dataset = pd.read_parquet('../../data/processed/pooled_modelling.parquet')
X = dataset.drop(columns=['label', 'weight', 't1', 'ticker'], errors='ignore').select_dtypes(include='number')
y = dataset['label'].astype(int)
y_xgb = (y == 1).astype(int)
sample_weight = dataset['weight']
t1 = dataset['t1']

majority_baseline = float((y == y.mode()[0]).mean())
print(f'X shape: {X.shape}')
print(f'majority-class baseline accuracy: {majority_baseline:.4f}')

pkf = PurgedKFold(n_splits=5, t1=t1, pct_embargo=0.01)

## 2. Search distributions

In [ ]:
rf_dist = {
    'n_estimators':      [100, 200, 500],
    'max_depth':         [3, 5, 7, 10],
    'min_samples_leaf':  [5, 10, 20, 30],
    'max_features':      ['sqrt', 0.5, 1.0],
    'class_weight':      ['balanced_subsample', None],
}
xgb_dist = {
    'n_estimators':     [100, 200, 500],
    'max_depth':        [2, 3, 5, 7],
    'learning_rate':    [0.01, 0.05, 0.1, 0.2],
    'subsample':        [0.5, 0.7, 0.8, 1.0],
    'colsample_bytree': [0.5, 0.7, 0.8, 1.0],
    'gamma':            [0.0, 0.1, 0.5],
    'reg_lambda':       [0.1, 1.0, 10.0],
}
N_ITER = 25

## 3. Run randomized search

In [ ]:
rf_base = RandomForestClassifier(random_state=RNG, n_jobs=-1)
xgb_base = XGBClassifier(random_state=RNG, n_jobs=-1, eval_metric='logloss')

rf_search = purged_random_search(
    clf=rf_base, X=X, y=y, sample_weight=sample_weight,
    param_dist=rf_dist, cv=pkf, n_iter=N_ITER,
    scoring='accuracy', random_state=RNG,
)
print(f"RF  best: mean={rf_search['best_mean_score']:.4f} std={rf_search['best_std_score']:.4f}")
print('  params:', rf_search['best_params'])

In [ ]:
xgb_search = purged_random_search(
    clf=xgb_base, X=X, y=y_xgb, sample_weight=sample_weight,
    param_dist=xgb_dist, cv=pkf, n_iter=N_ITER,
    scoring='accuracy', random_state=RNG,
)
print(f"XGB best: mean={xgb_search['best_mean_score']:.4f} std={xgb_search['best_std_score']:.4f}")
print('  params:', xgb_search['best_params'])

## 4. Trial summaries

In [ ]:
rf_log = rf_search['log'].assign(model='RF')
xgb_log = xgb_search['log'].assign(model='XGB')
tuning_log = pd.concat([rf_log, xgb_log], ignore_index=True)

print('RF  trial mean_score:'); print(rf_log['mean_score'].describe().round(4))
print()
print('XGB trial mean_score:'); print(xgb_log['mean_score'].describe().round(4))

In [ ]:
fig, ax = plt.subplots(figsize=(9, 4))
ax.hist(rf_log['mean_score'],  bins=12, alpha=0.6, label='RF',  color='#2c7fb8')
ax.hist(xgb_log['mean_score'], bins=12, alpha=0.6, label='XGB', color='#e34a33')
ax.axvline(majority_baseline, color='k', linestyle='--', alpha=0.7,
           label=f'majority baseline = {majority_baseline:.3f}')
ax.axvline(rf_search['best_mean_score'],  color='#2c7fb8', linestyle=':',
           label=f"RF best = {rf_search['best_mean_score']:.3f}")
ax.axvline(xgb_search['best_mean_score'], color='#e34a33', linestyle=':',
           label=f"XGB best = {xgb_search['best_mean_score']:.3f}")
ax.set_xlabel('weighted purged CV mean accuracy')
ax.set_ylabel('# trials')
ax.set_title(f'Trial-score distribution ({N_ITER} trials per model)')
ax.legend()
plt.tight_layout()
plt.show()

## 5. 2D hyperparameter heatmap

In [ ]:
def heatmap_pivot(log_df, p1, p2):
    return log_df.pivot_table(index=p1, columns=p2, values='mean_score', aggfunc='mean')

rf_pivot = heatmap_pivot(rf_log, 'param_max_depth', 'param_min_samples_leaf')
xgb_pivot = heatmap_pivot(xgb_log, 'param_max_depth', 'param_learning_rate')

fig, axes = plt.subplots(1, 2, figsize=(12, 4))
for ax, piv, title in [(axes[0], rf_pivot,  'RF: max_depth × min_samples_leaf'),
                       (axes[1], xgb_pivot, 'XGB: max_depth × learning_rate')]:
    im = ax.imshow(piv.values, aspect='auto', cmap='viridis')
    ax.set_xticks(range(len(piv.columns))); ax.set_xticklabels(piv.columns)
    ax.set_yticks(range(len(piv.index)));   ax.set_yticklabels(piv.index)
    ax.set_xlabel(piv.columns.name); ax.set_ylabel(piv.index.name); ax.set_title(title)
    valid = piv.values[~np.isnan(piv.values)]
    cutoff = valid.mean() if len(valid) else 0
    for i in range(piv.shape[0]):
        for j in range(piv.shape[1]):
            v = piv.values[i, j]
            if not np.isnan(v):
                ax.text(j, i, f'{v:.2f}', ha='center', va='center',
                        color='white' if v < cutoff else 'black', fontsize=8)
    plt.colorbar(im, ax=ax)
plt.tight_layout(); plt.show()

## 6. Deflated Sharpe across **CV trials** (not strategy returns)

Caveat in plain English: this is the AFML deflator applied to `mean / std` of fold accuracies for each tuning trial, treating those as in-sample Sharpe-like statistics. It corrects for selection bias across the 25 trials. It tells us nothing about live trading risk — that requires the full backtesting pipeline.

In [ ]:
rf_fold_arrays  = [t['fold_scores'] for t in rf_search['trials']]
xgb_fold_arrays = [t['fold_scores'] for t in xgb_search['trials']]

rf_dsr  = deflated_sharpe_ratio_for_trials(rf_fold_arrays,  best_idx=rf_search['best_idx'])
xgb_dsr = deflated_sharpe_ratio_for_trials(xgb_fold_arrays, best_idx=xgb_search['best_idx'])

dsr_table = pd.DataFrame([rf_dsr, xgb_dsr], index=['RF', 'XGB']).round(4)
dsr_table

## 7. Persist

In [ ]:
best_params = {
    'rf':  {'params': rf_search['best_params'],
            'mean_score': rf_search['best_mean_score'],
            'std_score':  rf_search['best_std_score'],
            'dsr_cv_trials': rf_dsr},
    'xgb': {'params': xgb_search['best_params'],
            'mean_score': xgb_search['best_mean_score'],
            'std_score':  xgb_search['best_std_score'],
            'dsr_cv_trials': xgb_dsr},
    'meta': {
        'n_iter_per_model': N_ITER,
        'cv':               'PurgedKFold(n_splits=5, pct_embargo=0.01)',
        'scoring':          'accuracy (weighted)',
        'baseline_majority': float(majority_baseline),
        'dsr_caveat': ('DSR here is over CV trial sharpes (in-sample), '
                       'not strategy-return sharpes; that belongs to '
                       'the backtesting stage.'),
    },
}

os.makedirs('../models', exist_ok=True)
with open('../../models/best_params.json', 'w') as f:
    json.dump(best_params, f, indent=2,
              default=lambda o: int(o) if isinstance(o, np.integer)
                                 else float(o) if isinstance(o, np.floating)
                                 else str(o))

tuning_log_to_save = tuning_log.copy()
for col in tuning_log_to_save.select_dtypes(include='object').columns:
    tuning_log_to_save[col] = tuning_log_to_save[col].astype(str)
tuning_log_to_save.to_parquet('../../data/processed/tuning_log.parquet')

print('Saved:')
print('  ../models/best_params.json')
print('  ../data/processed/tuning_log.parquet')

## 8. Notes

- 25 trials per model is a deliberately modest budget; the dataset is   small and a wider sweep would just amplify multiple-testing risk.
- A high `dsr_cv_trials.dsr` near 1 indicates the best trial sits   far above what would be expected from selection alone across 25   configs — but again, that is *in-sample CV stability*, not a   guarantee that the model beats baseline.
- Whether the tuned model beats the majority baseline is reported   honestly in notebook 08's final summary table.